# DNN Assignment - 2
# M. Tech. in Artificial Intelligence & Machine Learning
### Work Integrated Learning Programmes Division
### I Semester 2026-27

---

### **Student & Course Information**
* **Student Name:** Aravind Jarpala
* **Roll Number / Student ID:** 2026aj05462
* **Course Title:** Deep Neural Networks
* **Course Code:** AIMLZG511
---

### **Assignment Overview**

1. **RNN, LSTM and GRU**
2. **Transformer from scratch**

---

In [ ]:
import os
import sys
import random
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("Python version:", sys.version)
print("PyTorch version:", torch.__version__)
print("NumPy version:", np.__version__)
print("Pandas version:", pd.__version__)
print("Scikit-learn version:", sklearn.__version__)

print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not detected. Check the runtime settings.")

In [ ]:
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

print("Random seed configuration completed.")

##### Set Project directories

In [ ]:
columns = [
    "problem",
    "model",
    "seed",
    "seq_len",
    "test_acc",
    "macro_f1",
    "test_mse",
    "params",
    "sec_per_epoch",
    "val_loss_nats",
    "bits_per_char",
    "notes"
]

results = pd.DataFrame(columns=columns)

results.to_csv("results.csv", index=False)

print("Results tracker initialized.")
print(results.columns.tolist())

### Tasks: 1. Data Pipeline

##### Downlaod and extract the UCI HAR Dataset

In [ ]:
import zipfile
import urllib.request

data_dir = "data/har"
os.makedirs(data_dir, exist_ok=True)

url = (
    "https://archive.ics.uci.edu/static/public/240/"
    "human+activity+recognition+using+smartphones.zip"
)

archive_path = os.path.join(data_dir, "har.zip")

if not os.path.exists(archive_path):
    print("Downloading UCI HAR dataset...")
    urllib.request.urlretrieve(url, archive_path)

print("Extracting the downloaded archive...")

with zipfile.ZipFile(archive_path, "r") as zip_ref:
    zip_ref.extractall(data_dir)

# Extract the nested archive if it exists.
nested_zip = os.path.join(data_dir, "UCI HAR Dataset.zip")

if os.path.exists(nested_zip):
    with zipfile.ZipFile(nested_zip, "r") as zip_ref:
        zip_ref.extractall(data_dir)

dataset_path = os.path.join(data_dir, "UCI HAR Dataset")

assert os.path.isdir(dataset_path), "UCI HAR Dataset folder not found!"

print("Dataset extracted successfully.")
print("Dataset path:", dataset_path)
print("Available folders:", os.listdir(dataset_path))

In [ ]:
train_path = os.path.join(dataset_path, "train")
test_path = os.path.join(dataset_path, "test")

print("Training folders:", os.listdir(train_path))
print("Test folders:", os.listdir(test_path))

print(
    "Inertial signal files:",
    len(os.listdir(os.path.join(train_path, "Inertial Signals")))
)

In [ ]:
import shutil
project_dir = "/content/dnn_rnn_lstm_gru_and_transformer"

os.makedirs(project_dir, exist_ok=True)

# Move the already downloaded dataset into the project folder.
source_data = "/content/data/har"
target_data = os.path.join(project_dir, "data", "har")

if os.path.exists(source_data) and not os.path.exists(target_data):
    os.makedirs(os.path.dirname(target_data), exist_ok=True)
    shutil.move(source_data, target_data)

# Create folders for both problems.
os.makedirs(os.path.join(project_dir, "problem1", "figures"), exist_ok=True)
os.makedirs(os.path.join(project_dir, "problem2", "figures"), exist_ok=True)

os.chdir(project_dir)

print("Project directory:", os.getcwd())
print("Project contents:", os.listdir())

##### Inspect the HAR data

In [ ]:
dataset_path = "data/har/UCI HAR Dataset"

channel_names = [
    "body_acc_x", "body_acc_y", "body_acc_z",
    "body_gyro_x", "body_gyro_y", "body_gyro_z",
    "total_acc_x", "total_acc_y", "total_acc_z"
]

def load_har_split(split):
    split_path = os.path.join(dataset_path, split, "Inertial Signals")

    signals = []

    for channel in channel_names:
        file_path = os.path.join(
            split_path,
            f"{channel}_{split}.txt"
        )

        channel_data = np.loadtxt(file_path)
        signals.append(channel_data)

    # Each channel has shape (number of samples, 128).
    # Stack channels along the last axis.
    X = np.stack(signals, axis=-1)

    y = np.loadtxt(
        os.path.join(dataset_path, split, f"y_{split}.txt")
    ).astype(np.int64)

    subjects = np.loadtxt(
        os.path.join(dataset_path, split, f"subject_{split}.txt")
    ).astype(np.int64)

    return X.astype(np.float32), y, subjects


X_train, y_train, subject_train = load_har_split("train")
X_test, y_test, subject_test = load_har_split("test")

print("Training data shape:", X_train.shape)
print("Training labels shape:", y_train.shape)
print("Training subjects shape:", subject_train.shape)

print("Test data shape:", X_test.shape)
print("Test labels shape:", y_test.shape)
print("Test subjects shape:", subject_test.shape)

print("Unique activity labels:", np.unique(y_train))
print("Unique training subjects:", np.unique(subject_train))

#### Split training subjects into training and validation groups.


In [ ]:
def split_subjects(subject_ids, seed):
    unique_subjects = np.unique(subject_ids)

    rng = np.random.default_rng(seed)

    validation_subjects = rng.choice(
        unique_subjects,
        size=4,
        replace=False
    )

    training_subjects = np.setdiff1d(
        unique_subjects,
        validation_subjects
    )

    return training_subjects, validation_subjects


# Temporary seed for checking the pipeline.
split_seed = 42

train_subjects, val_subjects = split_subjects(
    subject_train,
    split_seed
)

# Select windows based on subject membership.
train_mask = np.isin(subject_train, train_subjects)
val_mask = np.isin(subject_train, val_subjects)

X_train_split = X_train[train_mask]
y_train_split = y_train[train_mask]

X_val = X_train[val_mask]
y_val = y_train[val_mask]

print("Training subjects:", train_subjects)
print("Validation subjects:", val_subjects)

print("\nTraining data:", X_train_split.shape)
print("Training labels:", y_train_split.shape)

print("\nValidation data:", X_val.shape)
print("Validation labels:", y_val.shape)

##### Adding assertions to make sure no subject appears in both groups and all training windows

In [ ]:
# Check that the subject groups do not overlap.
assert len(np.intersect1d(train_subjects, val_subjects)) == 0

# Check that all original training subjects are accounted for.
assert len(train_subjects) == 17
assert len(val_subjects) == 4

# Check that the two partitions contain all original windows.
assert len(X_train_split) + len(X_val) == len(X_train)

# Check that no subject crosses the split.
assert set(np.unique(subject_train[train_mask])).isdisjoint(
    set(np.unique(subject_train[val_mask]))
)

print("All subject-wise split checks passed!")

##### Normalization

In [ ]:
# Calculate statistics only from the training partition.
channel_mean = X_train_split.mean(axis=(0, 1))
channel_std = X_train_split.std(axis=(0, 1))

# Avoid division by zero.
channel_std = np.maximum(channel_std, 1e-8)

# Apply the same statistics to all partitions.
X_train_normalized = (
    X_train_split - channel_mean
) / channel_std

X_val_normalized = (
    X_val - channel_mean
) / channel_std

X_test_normalized = (
    X_test - channel_mean
) / channel_std

print("Channel means:", channel_mean)
print("Channel standard deviations:", channel_std)
# Verify that normalized training data has approximately
# zero mean and unit standard deviation for every channel.

normalized_mean = X_train_normalized.mean(axis=(0, 1))
normalized_std = X_train_normalized.std(axis=(0, 1))

# A tolerance of 2e-4 accounts for float32 numerical precision.
assert np.allclose(
    normalized_mean,
    0,
    atol=2e-4
)

assert np.allclose(
    normalized_std,
    1,
    atol=2e-4
)

# Check that validation and test data have the same shape
# and use the same normalization statistics.
assert X_val_normalized.shape == X_val.shape
assert X_test_normalized.shape == X_test.shape

print("Normalization checks passed!")
print("Maximum absolute mean deviation:", np.abs(normalized_mean).max())
print("Maximum standard deviation deviation:", np.abs(normalized_std - 1).max())

##### Plot the HAR activity class distribution

In [ ]:
activity_info = np.loadtxt(
    os.path.join(dataset_path, "activity_labels.txt"),
    dtype=str
)

activity_names = {
    int(row[0]): row[1].replace("_", " ").title()
    for row in activity_info
}

# Activity IDs in the dataset
class_ids = np.arange(1, 7)

# Count the number of windows belonging to each activity
train_counts = [
    np.sum(y_train_split == class_id)
    for class_id in class_ids
]

val_counts = [
    np.sum(y_val == class_id)
    for class_id in class_ids
]

# Prepare the bar chart
x = np.arange(len(class_ids))
bar_width = 0.38

plt.figure(figsize=(11, 5))

plt.bar(
    x - bar_width / 2,
    train_counts,
    bar_width,
    label="Training"
)

plt.bar(
    x + bar_width / 2,
    val_counts,
    bar_width,
    label="Validation"
)

plt.xticks(
    x,
    [activity_names[class_id] for class_id in class_ids],
    rotation=20,
    ha="right"
)

plt.xlabel("Activity")
plt.ylabel("Number of windows")
plt.title("HAR Activity Distribution: Training vs Validation")
plt.legend()
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()

# Save the figure for the final report
figure_path = "problem1/figures/har_class_balance.png"
plt.savefig(figure_path, dpi=200, bbox_inches="tight")
plt.show()

# Verify that all samples are represented
assert sum(train_counts) == len(y_train_split)
assert sum(val_counts) == len(y_val)

print("Training samples per activity:")
for class_id, count in zip(class_ids, train_counts):
    print(f"{activity_names[class_id]}: {count}")

print("\nValidation samples per activity:")
for class_id, count in zip(class_ids, val_counts):
    print(f"{activity_names[class_id]}: {count}")

print("\nFigure saved to:", figure_path)


##### Adding problem generator

In [ ]:
def make_adding(n, T, seed):
  """
  Generate the Adding Problem dataset.

  Parameters:
      n    : Number of sequences
      T    : Sequence length
      seed : Random seed

  Returns:
      x : Input sequences of shape (n, T, 2)
      y : Target sums of shape (n,)
  """
  rng = np.random.default_rng(seed)

  # Channel 0 contains random values.
  # Channel 1 contains two marker positions.
  x = np.zeros((n, T, 2), dtype=np.float32)

  # Generate random values in channel 0
  x[:, :, 0] = rng.random((n, T))

  # Select one position from each half
  i1 = rng.integers(0, T // 2, n)
  i2 = rng.integers(T // 2, T, n)

  # Place the two markers in channel 1
  x[np.arange(n), i1, 1] = 1
  x[np.arange(n), i2, 1] = 1

  # Target is the sum of the two marked values
  y = x[np.arange(n), i1, 0] + x[np.arange(n), i2, 0]

  return x, y.astype(np.float32)

##### Validate the generator

In [ ]:
X_adding_test, y_adding_test = make_adding(
    n=1000,
    T=50,
    seed=42
)

# Check input and target dimensions
assert X_adding_test.shape == (1000, 50, 2)
assert y_adding_test.shape == (1000,)

# Extract marker channel
markers = X_adding_test[:, :, 1]

# Every sequence must contain exactly two markers
assert np.all(markers.sum(axis=1) == 2)

# First-half and second-half marker checks
first_half_markers = markers[:, :25].sum(axis=1)
second_half_markers = markers[:, 25:].sum(axis=1)

assert np.all(first_half_markers == 1)
assert np.all(second_half_markers == 1)

# Independently calculate the target from marked values
calculated_targets = (
    X_adding_test[:, :, 0] * markers
).sum(axis=1)

assert np.allclose(
    y_adding_test,
    calculated_targets,
    atol=1e-6
)

# Check target range
assert np.all(y_adding_test >= 0)
assert np.all(y_adding_test <= 2)

print("Adding Problem generator validation passed!")
print("Input shape:", X_adding_test.shape)
print("Target shape:", y_adding_test.shape)
print("Markers per sequence:", np.unique(markers.sum(axis=1)))
print("First-half markers:", np.unique(first_half_markers))
print("Second-half markers:", np.unique(second_half_markers))
print("Target range:", y_adding_test.min(), "to", y_adding_test.max())
print("First five targets:", y_adding_test[:5])


##### Generate the actual data

In [ ]:
sequence_lengths = [50, 100, 200, 400]
adding_data = {}

for T in sequence_lengths:

  # Training dataset
  X_train_add, y_train_add = make_adding(
      n=10000, T=T, seed=0
  )

  # Validation dataset
  X_val_add, y_val_add = make_adding(
      n=1000, T=T, seed=1
  )

  # Test dataset
  X_test_add, y_test_add = make_adding(
      n=1000, T=T, seed=2
  )

  adding_data[T] = {
      "X_train": X_train_add,
      "y_train": y_train_add,
      "X_val": X_val_add,
      "y_val": y_val_add,
      "X_test": X_test_add,
      "y_test": y_test_add
  }

  print(f"\nSequence length T = {T}")
  print("Training:", X_train_add.shape, y_train_add.shape)
  print("Validation:", X_val_add.shape, y_val_add.shape)
  print("Testing:", X_test_add.shape, y_test_add.shape)

print("\nAll Adding Problem datasets generated successfully!")

##### Calculate baseline MSE

In [ ]:
baseline_results = {}

for T in sequence_lengths:
  y_test = adding_data[T]["y_test"]

  # Constant prediction: always predict 1.0
  baseline_predictions = np.ones_like(y_test)

  baseline_mse = np.mean(
      (y_test - baseline_predictions) ** 2
  )

  baseline_results[T] = baseline_mse

  print(
      f"T = {T:3d} | "
      f"Baseline Test MSE = {baseline_mse:.6f}"
  )

print("\nTheoretical baseline MSE:", 1 / 6)

##2. Implement Cells from scratch

### 2(a): Implement Vanilla RNN from scratch

In [ ]:
class MyRNNCell(torch.nn.Module):
  def __init__(self, input_size, hidden_size):
    super().__init__()
    self.input_size = input_size
    self.hidden_size = hidden_size

    # Input-to-hidden weights: (hidden_size, input_size)
    self.weight_ih = torch.nn.Parameter(
        torch.empty(hidden_size, input_size)
    )

    # Hidden-to-hidden weights: (hidden_size, hidden_size)
    self.weight_hh = torch.nn.Parameter(
        torch.empty(hidden_size, hidden_size)
    )

    # Combined bias: (hidden_size,)
    self.bias = torch.nn.Parameter(
        torch.empty(hidden_size)
    )

    # Initialize parameters
    torch.nn.init.xavier_uniform_(self.weight_ih)
    torch.nn.init.orthogonal_(self.weight_hh)
    torch.nn.init.zeros_(self.bias)

  def forward(self, x_t, h_prev):
    # Input contribution
    input_part = torch.nn.functional.linear(
        x_t, self.weight_ih
    )

    # Previous hidden-state contribution
    hidden_part = torch.nn.functional.linear(
        h_prev, self.weight_hh
    )

    # Combine contributions and apply tanh
    h_t = torch.tanh(
        input_part + hidden_part + self.bias
    )

    return h_t

##### Testing the implementation

In [ ]:
# Set dimensions
input_size = 9
hidden_size = 64
batch_size = 32

# Create custom RNN cell
my_rnn = MyRNNCell(input_size, hidden_size)

# Random input and previous hidden state
x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)

# Forward pass
h_new = my_rnn(x_t, h_prev)

# Parameter count
parameter_count = sum(
    p.numel() for p in my_rnn.parameters()
)

# Validation
assert h_new.shape == (batch_size, hidden_size)
assert parameter_count == 4736

print("Output shape:", h_new.shape)
print("Trainable parameters:", parameter_count)
print("All MyRNNCell shape and parameter checks passed!")

##### Verfying against pytorch reference implementation

In [ ]:
# PyTorch reference RNN
reference_rnn = torch.nn.RNN(
    input_size=9,
    hidden_size=64,
    nonlinearity="tanh",
    batch_first=True,
    bias=True
)

# Copy reference weights into the custom cell
with torch.no_grad():
  my_rnn.weight_ih.copy_(reference_rnn.weight_ih_l0)
  my_rnn.weight_hh.copy_(reference_rnn.weight_hh_l0)

  my_rnn.bias.copy_(
      reference_rnn.bias_ih_l0 +
      reference_rnn.bias_hh_l0
  )

# Create random input sequence
x_sequence = torch.randn(32, 5, 9)
h_initial = torch.randn(1, 32, 64)

# Reference output
reference_output, reference_hidden = reference_rnn(
    x_sequence, h_initial
)

# Custom RNN: process one timestep at a time
custom_hidden = h_initial.squeeze(0)
custom_outputs = []

for t in range(x_sequence.shape[1]):
  custom_hidden = my_rnn(
      x_sequence[:, t, :],
      custom_hidden
  )
  custom_outputs.append(custom_hidden)

custom_output = torch.stack(custom_outputs, dim=1)

# Compare outputs
max_difference = torch.max(
    torch.abs(reference_output - custom_output)
).item()

print("Reference output shape:", reference_output.shape)
print("Custom output shape:", custom_output.shape)
print("Maximum absolute difference:", max_difference)

assert max_difference < 1e-5

print("Numerical equivalence test passed!")

### 2(a): Implement LSTM Cell from scratch

In [ ]:

class MyLSTMCell(torch.nn.Module):
  def __init__(self, input_size, hidden_size):
    super().__init__()
    self.input_size = input_size
    self.hidden_size = hidden_size

    # Weight matrices for all four gates: input, forget, candidate, output
    self.weight_ih = torch.nn.Parameter(
        torch.empty(4 * hidden_size, input_size)
    )
    self.weight_hh = torch.nn.Parameter(
        torch.empty(4 * hidden_size, hidden_size)
    )
    self.bias = torch.nn.Parameter(
        torch.empty(4 * hidden_size)
    )

    torch.nn.init.xavier_uniform_(self.weight_ih)
    torch.nn.init.orthogonal_(self.weight_hh)
    torch.nn.init.zeros_(self.bias)

  def forward(self, x_t, h_prev, c_prev):
    # Calculate the input and recurrent contributions
    gates = (
        torch.nn.functional.linear(x_t, self.weight_ih)
        + torch.nn.functional.linear(h_prev, self.weight_hh)
        + self.bias
    )

    # Split the combined result into four gates (PyTorch order: i, f, g, o)
    i_gate, f_gate, g_gate, o_gate = gates.chunk(4, dim=1)

    # Apply the corresponding activation functions
    i_gate = torch.sigmoid(i_gate)
    f_gate = torch.sigmoid(f_gate)
    g_gate = torch.tanh(g_gate)
    o_gate = torch.sigmoid(o_gate)

    # Update the cell state and hidden state
    c_t = f_gate * c_prev + i_gate * g_gate
    h_t = o_gate * torch.tanh(c_t)

    return h_t, c_t

##### Verify output and parameter count

In [ ]:

input_size = 9
hidden_size = 64
batch_size = 32

my_lstm = MyLSTMCell(input_size, hidden_size)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)
c_prev = torch.randn(batch_size, hidden_size)

h_new, c_new = my_lstm(x_t, h_prev, c_prev)

parameter_count = sum(p.numel() for p in my_lstm.parameters())

print("Hidden state shape:", h_new.shape)
print("Cell state shape:", c_new.shape)
print("Trainable parameters:", parameter_count)

assert h_new.shape == (batch_size, hidden_size)
assert c_new.shape == (batch_size, hidden_size)
assert parameter_count == 18944

print("All MyLSTMCell shape and parameter checks passed!")


##### Verify against pytorch reference implementation

In [ ]:
reference_lstm = torch.nn.LSTM(
    input_size=9,
    hidden_size=64,
    batch_first=True,
    bias=True
)

with torch.no_grad():
  my_lstm.weight_ih.copy_(reference_lstm.weight_ih_l0)
  my_lstm.weight_hh.copy_(reference_lstm.weight_hh_l0)

  my_lstm.bias.copy_(
      reference_lstm.bias_ih_l0 + reference_lstm.bias_hh_l0
  )

# Generate a sample sequence
x_sequence = torch.randn(32, 5, 9)

h_initial = torch.randn(1, 32, 64)
c_initial = torch.randn(1, 32, 64)

# PyTorch reference output
reference_output, (reference_h, reference_c) = reference_lstm(
    x_sequence, (h_initial, c_initial)
)

# Custom LSTM output
custom_h = h_initial.squeeze(0)
custom_c = c_initial.squeeze(0)

custom_outputs = []

for t in range(x_sequence.shape[1]):
  custom_h, custom_c = my_lstm(
      x_sequence[:, t, :],
      custom_h,
      custom_c
  )
  custom_outputs.append(custom_h)

custom_output = torch.stack(custom_outputs, dim=1)

# Compare sequence outputs and final states
output_difference = torch.max(
    torch.abs(reference_output - custom_output)
).item()

hidden_difference = torch.max(
    torch.abs(reference_h.squeeze(0) - custom_h)
).item()

cell_difference = torch.max(
    torch.abs(reference_c.squeeze(0) - custom_c)
).item()

print("Reference output shape:", reference_output.shape)
print("Custom output shape:", custom_output.shape)
print("Maximum output difference:", output_difference)
print("Maximum hidden-state difference:", hidden_difference)
print("Maximum cell-state difference:", cell_difference)

assert output_difference < 1e-5
assert hidden_difference < 1e-5
assert cell_difference < 1e-5

print("LSTM numerical equivalence test passed!")

### 2(c): Implement GRU Cell from scratch

In [ ]:
class MyGRUCell(torch.nn.Module):
  def __init__(self, input_size, hidden_size):
    super().__init__()
    self.input_size = input_size
    self.hidden_size = hidden_size

    # Weight matrices for reset, update and candidate gates
    self.weight_ih = torch.nn.Parameter(
        torch.empty(3 * hidden_size, input_size)
    )
    self.weight_hh = torch.nn.Parameter(
        torch.empty(3 * hidden_size, hidden_size)
    )

    # Separate biases to match PyTorch GRU computation
    self.bias_ih = torch.nn.Parameter(
        torch.empty(3 * hidden_size)
    )
    self.bias_hh = torch.nn.Parameter(
        torch.empty(3 * hidden_size)
    )

    torch.nn.init.xavier_uniform_(self.weight_ih)
    torch.nn.init.orthogonal_(self.weight_hh)
    torch.nn.init.zeros_(self.bias_ih)
    torch.nn.init.zeros_(self.bias_hh)

  def forward(self, x_t, h_prev):
    # Input and recurrent contributions
    input_gates = torch.nn.functional.linear(
        x_t, self.weight_ih, self.bias_ih
    )

    hidden_gates = torch.nn.functional.linear(
        h_prev, self.weight_hh, self.bias_hh
    )

    # Split gates in PyTorch order: reset, update, new
    input_r, input_z, input_n = input_gates.chunk(3, dim=1)
    hidden_r, hidden_z, hidden_n = hidden_gates.chunk(3, dim=1)

    # Reset and update gates
    r_gate = torch.sigmoid(input_r + hidden_r)
    z_gate = torch.sigmoid(input_z + hidden_z)

    # Candidate hidden state
    n_gate = torch.tanh(input_n + r_gate * hidden_n)

    # Final hidden state
    h_t = (1 - z_gate) * n_gate + z_gate * h_prev

    return h_t

##### Verify output and parameter counts

In [ ]:
input_size = 9
hidden_size = 64
batch_size = 32

my_gru = MyGRUCell(input_size, hidden_size)

x_t = torch.randn(batch_size, input_size)
h_prev = torch.randn(batch_size, hidden_size)

h_new = my_gru(x_t, h_prev)

parameter_count = sum(p.numel() for p in my_gru.parameters())

print("Hidden state shape:", h_new.shape)
print("Trainable parameters:", parameter_count)

assert h_new.shape == (batch_size, hidden_size)
assert parameter_count == 14400

print("All MyGRUCell shape and parameter checks passed!")

##### Verify against pytorch reference implementation

In [ ]:
reference_gru = torch.nn.GRU(
    input_size=9,
    hidden_size=64,
    batch_first=True,
    bias=True
)

with torch.no_grad():
  my_gru.weight_ih.copy_(reference_gru.weight_ih_l0)
  my_gru.weight_hh.copy_(reference_gru.weight_hh_l0)

  my_gru.bias_ih.copy_(reference_gru.bias_ih_l0)
  my_gru.bias_hh.copy_(reference_gru.bias_hh_l0)

# Generate a sample sequence
x_sequence = torch.randn(32, 5, 9)
h_initial = torch.randn(1, 32, 64)

# PyTorch reference output
reference_output, reference_hidden = reference_gru(
    x_sequence, h_initial
)

# Custom GRU output
custom_hidden = h_initial.squeeze(0)
custom_outputs = []

for t in range(x_sequence.shape[1]):
  custom_hidden = my_gru(
      x_sequence[:, t, :],
      custom_hidden
  )
  custom_outputs.append(custom_hidden)

custom_output = torch.stack(custom_outputs, dim=1)

# Compare outputs and final hidden state
output_difference = torch.max(
    torch.abs(reference_output - custom_output)
).item()

hidden_difference = torch.max(
    torch.abs(reference_hidden.squeeze(0) - custom_hidden)
).item()

print("Reference output shape:", reference_output.shape)
print("Custom output shape:", custom_output.shape)
print("Maximum output difference:", output_difference)
print("Maximum hidden-state difference:", hidden_difference)

assert output_difference < 1e-5
assert hidden_difference < 1e-5

print("GRU numerical equivalence test passed!")

## Task 3: HAR Classification

#### Prepare data independently for each seed

In [ ]:
def prepare_har_data(seed):
  # Create a subject-wise training/validation split
  train_subjects, val_subjects = split_subjects(subject_train, seed)

  train_mask = np.isin(subject_train, train_subjects)
  val_mask = np.isin(subject_train, val_subjects)

  X_train_seed = X_train[train_mask]
  y_train_seed = y_train[train_mask]

  X_val_seed = X_train[val_mask]
  y_val_seed = y_train[val_mask]

  # Calculate normalization statistics using training data only
  channel_mean = X_train_seed.mean(axis=(0, 1))
  channel_std = X_train_seed.std(axis=(0, 1))

  channel_std = np.maximum(channel_std, 1e-8)

  # Apply the same statistics to all three datasets
  X_train_norm = (X_train_seed - channel_mean) / channel_std
  X_val_norm = (X_val_seed - channel_mean) / channel_std
  X_test_norm = (X_test - channel_mean) / channel_std


  # Convert labels from 1-6 to 0-5
  y_train_seed = y_train_seed - 1
  y_val_seed = y_val_seed - 1

  y_test_original = np.loadtxt(
    os.path.join(dataset_path, "test", "y_test.txt")
  ).astype(np.int64)
  # y_test_seed = y_test - 1
  y_test_seed = y_test_original - 1

  # Check dimensions and label ranges
  assert X_train_norm.shape[1:] == (128, 9)
  assert X_val_norm.shape[1:] == (128, 9)
  assert X_test_norm.shape[1:] == (128, 9)

  assert set(np.unique(y_train_seed)) == set(range(6))
  assert set(np.unique(y_val_seed)) == set(range(6))
  assert set(np.unique(y_test_seed)) == set(range(6))

  # Verify training-only normalization
  train_channel_means = X_train_norm.mean(axis=(0, 1))
  train_channel_stds = X_train_norm.std(axis=(0, 1))

  print(f"\nNormalization check for seed {seed}:")
  print("Channel means:", train_channel_means)
  print("Channel standard deviations:", train_channel_stds)

  print("Maximum absolute mean:", np.max(np.abs(train_channel_means)))
  print("Maximum std deviation error:", np.max(np.abs(train_channel_stds - 1)))
  return {
      "X_train": X_train_norm.astype(np.float32),
      "y_train": y_train_seed.astype(np.int64),
      "X_val": X_val_norm.astype(np.float32),
      "y_val": y_val_seed.astype(np.int64),
      "X_test": X_test_norm.astype(np.float32),
      "y_test": y_test_seed.astype(np.int64),
      "train_subjects": train_subjects,
      "val_subjects": val_subjects,
      "channel_mean": channel_mean,
      "channel_std": channel_std
  }


# Validate the pipeline with all three seeds
seed_data = {}

for seed in [42, 123, 2026]:
  data = prepare_har_data(seed)
  seed_data[seed] = data

  print(f"\nSeed: {seed}")
  print("Training shape:", data["X_train"].shape)
  print("Validation shape:", data["X_val"].shape)
  print("Test shape:", data["X_test"].shape)
  print("Training subjects:", data["train_subjects"])
  print("Validation subjects:", data["val_subjects"])

print("\nAll three seed-specific data pipelines passed!")

##### Create PyTorch datasets and DataLoaders

In [ ]:
from torch.utils.data import TensorDataset, DataLoader

BATCH_SIZE = 128

def create_har_loaders(data, seed):
  train_dataset = TensorDataset(
      torch.from_numpy(data["X_train"]),
      torch.from_numpy(data["y_train"])
  )

  val_dataset = TensorDataset(
      torch.from_numpy(data["X_val"]),
      torch.from_numpy(data["y_val"])
  )

  test_dataset = TensorDataset(
      torch.from_numpy(data["X_test"]),
      torch.from_numpy(data["y_test"])
  )

  # Use a dedicated generator for reproducible shuffling
  generator = torch.Generator()
  generator.manual_seed(seed)

  train_loader = DataLoader(
      train_dataset,
      batch_size=BATCH_SIZE,
      shuffle=True,
      generator=generator,
      num_workers=0,
      pin_memory=torch.cuda.is_available()
  )

  val_loader = DataLoader(
      val_dataset,
      batch_size=BATCH_SIZE,
      shuffle=False,
      num_workers=0,
      pin_memory=torch.cuda.is_available()
  )

  test_loader = DataLoader(
      test_dataset,
      batch_size=BATCH_SIZE,
      shuffle=False,
      num_workers=0,
      pin_memory=torch.cuda.is_available()
  )

  return train_loader, val_loader, test_loader


# Test the DataLoaders
train_loader, val_loader, test_loader = create_har_loaders(
    seed_data[42], 42
)

sample_inputs, sample_labels = next(iter(train_loader))

print("Input batch shape:", sample_inputs.shape)
print("Label batch shape:", sample_labels.shape)
print("Input dtype:", sample_inputs.dtype)
print("Label dtype:", sample_labels.dtype)
print("Number of training batches:", len(train_loader))

assert sample_inputs.shape == (128, 128, 9)
assert sample_labels.shape == (128,)
assert sample_inputs.dtype == torch.float32
assert sample_labels.dtype == torch.int64

print("DataLoader checks passed!")

##### Build the three HAR classifiers

In [ ]:
import torch.nn as nn

HIDDEN_SIZE = 64
INPUT_SIZE = 9
NUM_CLASSES = 6

class HARRecurrentClassifier(nn.Module):
  def __init__(self, model_type, input_size=9, hidden_size=64, num_classes=6):
    super().__init__()
    self.model_type = model_type.upper()

    if self.model_type == "RNN":
        self.recurrent = nn.RNN(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True,
            nonlinearity="tanh"
        )

    elif self.model_type == "LSTM":
        self.recurrent = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

    elif self.model_type == "GRU":
        self.recurrent = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            batch_first=True
        )

    else:
        raise ValueError("Model type must be RNN, LSTM or GRU")

    self.classifier = nn.Linear(hidden_size, num_classes)

  def forward(self, x):
    # Input shape: (batch_size, sequence_length, input_size)
    if self.model_type == "LSTM":
        _, (hidden, cell) = self.recurrent(x)
    else:
        _, hidden = self.recurrent(x)

    # Final hidden state of the last recurrent layer
    final_hidden = hidden[-1]

    # Convert the final representation into class logits
    logits = self.classifier(final_hidden)

    return logits

# Verify all three architectures
for model_name in ["RNN", "LSTM", "GRU"]:
  model = HARRecurrentClassifier(model_name)

  sample_output = model(sample_inputs)

  params = sum(p.numel() for p in model.parameters())

  print(f"\n{model_name}")
  print("Output shape:", sample_output.shape)
  print("Trainable parameters:", params)

  assert sample_output.shape == (128, 6)

print("\nAll three classifier shape checks passed!")

##### Shared training and evaluation functions

In [ ]:

import time
import copy

from sklearn.metrics import (accuracy_score, f1_score, confusion_matrix)

LEARNING_RATE = 1e-3
EPOCHS = 15

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", device)

def evaluate_model(model, data_loader, criterion):
  model.eval()

  total_loss = 0.0
  all_predictions = []
  all_labels = []

  with torch.no_grad():
    for inputs, labels in data_loader:
      inputs = inputs.to(device)
      labels = labels.to(device)

      logits = model(inputs)
      loss = criterion(logits, labels)

      total_loss += loss.item() * labels.size(0)

      predictions = torch.argmax(logits, dim=1)

      all_predictions.extend(predictions.cpu().numpy())
      all_labels.extend(labels.cpu().numpy())

  average_loss = total_loss / len(data_loader.dataset)

  accuracy = accuracy_score(all_labels, all_predictions)
  macro_f1 = f1_score(
      all_labels,
      all_predictions,
      average="macro"
  )

  matrix = confusion_matrix(
      all_labels,
      all_predictions,
      labels=list(range(NUM_CLASSES))
  )

  return average_loss, accuracy, macro_f1, matrix


def train_one_model(model, train_loader, val_loader):
  model = model.to(device)
  criterion = nn.CrossEntropyLoss()

  optimizer = torch.optim.Adam(
      model.parameters(),
      lr=LEARNING_RATE
  )

  best_val_f1 = -1.0
  best_epoch = 0
  best_state = None

  epoch_times = []
  history = []

  for epoch in range(EPOCHS):
      model.train()

      running_loss = 0.0

      if device.type == "cuda":
          torch.cuda.synchronize()

      start_time = time.perf_counter()

      for inputs, labels in train_loader:
          inputs = inputs.to(device)
          labels = labels.to(device)

          optimizer.zero_grad(set_to_none=True)

          logits = model(inputs)
          loss = criterion(logits, labels)

          loss.backward()
          optimizer.step()

          running_loss += loss.item() * labels.size(0)

      if device.type == "cuda":
          torch.cuda.synchronize()

      epoch_time = time.perf_counter() - start_time
      epoch_times.append(epoch_time)

      train_loss = running_loss / len(train_loader.dataset)

      val_loss, val_acc, val_f1, _ = evaluate_model(
          model, val_loader, criterion
      )

      history.append({
          "epoch": epoch + 1,
          "train_loss": train_loss,
          "val_loss": val_loss,
          "val_accuracy": val_acc,
          "val_macro_f1": val_f1
      })

      # Keep the checkpoint with the best validation macro-F1
      if val_f1 > best_val_f1:
          best_val_f1 = val_f1
          best_epoch = epoch + 1
          best_state = copy.deepcopy(model.state_dict())

      print(
          f"Epoch [{epoch + 1:02d}/{EPOCHS}] "
          f"Train Loss: {train_loss:.4f} | "
          f"Val Loss: {val_loss:.4f} | "
          f"Val Accuracy: {val_acc:.4f} | "
          f"Val Macro-F1: {val_f1:.4f} | "
          f"Time: {epoch_time:.2f}s"
      )

  # Restore the best validation checkpoint
  model.load_state_dict(best_state)

  return model, history, best_epoch, best_val_f1, np.mean(epoch_times)

##### Runnning all 9 experiments

In [ ]:

import os

SEEDS = [42, 123, 2026]
MODEL_NAMES = ["RNN", "LSTM", "GRU"]

RESULT_COLUMNS = [
    "problem", "model", "seed", "seq_len",
    "test_acc", "macro_f1", "test_mse",
    "params", "sec_per_epoch",
    "val_loss_nats", "bits_per_char", "notes"
]

results_path = "results.csv"

# Create the file if it does not exist
if not os.path.exists(results_path):
    pd.DataFrame(columns=RESULT_COLUMNS).to_csv(
        results_path, index=False
    )

# Check that the existing results file has the required schema
existing_results = pd.read_csv(results_path)
assert list(existing_results.columns) == RESULT_COLUMNS

experiment_results = []
training_histories = {}

best_overall = {
    "val_macro_f1": -1.0,
    "model": None,
    "seed": None,
    "confusion_matrix": None
}

for seed in SEEDS:
    print("\n" + "=" * 65)
    print(f"EXPERIMENTS FOR SEED: {seed}")
    print("=" * 65)

    data = seed_data[seed]

    for model_name in MODEL_NAMES:
      print(f"\nTraining {model_name} | Seed {seed}")
      print("-" * 50)

      # Re-create loaders so each architecture gets the same
      # reproducible shuffled batch order for this seed
      train_loader, val_loader, test_loader = create_har_loaders(
          data, seed
      )

      # Reproduce model initialization for this run
      set_seed(seed)

      model = HARRecurrentClassifier(
          model_type=model_name,
          input_size=INPUT_SIZE,
          hidden_size=HIDDEN_SIZE,
          num_classes=NUM_CLASSES
      )

      parameter_count = sum(
          p.numel() for p in model.parameters()
      )

      # Train and select the best checkpoint using validation F1
      trained_model, history, best_epoch, best_val_f1, sec_per_epoch = (
          train_one_model(model, train_loader, val_loader)
      )

      # Evaluate the selected checkpoint on the official test set
      criterion = nn.CrossEntropyLoss()

      test_loss, test_acc, test_f1, test_cm = evaluate_model(
          trained_model, test_loader, criterion
      )

      print(f"\n{model_name} | Seed {seed} final test results")
      print(f"Best epoch: {best_epoch}")
      print(f"Best validation macro-F1: {best_val_f1:.4f}")
      print(f"Test accuracy: {test_acc:.4f}")
      print(f"Test macro-F1: {test_f1:.4f}")
      print(f"Parameters: {parameter_count}")
      print(f"Average training time/epoch: {sec_per_epoch:.2f}s")

      # Store the complete learning history
      training_histories[(model_name, seed)] = history

      # Prepare one row in the required results.csv format
      result_row = {
          "problem": "P1",
          "model": model_name,
          "seed": seed,
          "seq_len": 128,
          "test_acc": test_acc,
          "macro_f1": test_f1,
          "test_mse": "",
          "params": parameter_count,
          "sec_per_epoch": sec_per_epoch,
          "val_loss_nats": "",
          "bits_per_char": "",
          "notes": (
              f"best_epoch={best_epoch}; "
              f"best_val_macro_f1={best_val_f1:.4f}"
          )
      }

      experiment_results.append(result_row)

      # Append the result immediately after each completed run
      pd.DataFrame([result_row], columns=RESULT_COLUMNS).to_csv(
          results_path,
          mode="a",
          header=False,
          index=False
      )

      # Select the overall best run using validation macro-F1
      if best_val_f1 > best_overall["val_macro_f1"]:
          best_overall = {
              "val_macro_f1": best_val_f1,
              "model": model_name,
              "seed": seed,
              "confusion_matrix": test_cm.copy()
          }

print("\n" + "=" * 65)
print("ALL NINE EXPERIMENTS COMPLETED")
print("=" * 65)

print("Best validation-selected run:",
    best_overall["model"],
    "| Seed:",
    best_overall["seed"],
    "| Validation macro-F1:",
    round(best_overall["val_macro_f1"], 4)
)

#### Summarize mean and sample standard deviation

In [ ]:
results_df = pd.DataFrame(experiment_results)

summary_df = results_df.groupby("model").agg(
    accuracy_mean=("test_acc", "mean"),
    accuracy_std=("test_acc", "std"),
    macro_f1_mean=("macro_f1", "mean"),
    macro_f1_std=("macro_f1", "std"),
    parameters=("params", "first"),
    sec_per_epoch=("sec_per_epoch", "mean")
).reset_index()

summary_df["accuracy_mean_std"] = summary_df.apply(
    lambda row: f"{row['accuracy_mean']:.4f} ± {row['accuracy_std']:.4f}",
    axis=1
)

summary_df["macro_f1_mean_std"] = summary_df.apply(
    lambda row: f"{row['macro_f1_mean']:.4f} ± {row['macro_f1_std']:.4f}",
    axis=1
)

print("HAR Classification Results")
display(summary_df[["model",  "accuracy_mean_std", "macro_f1_mean_std", "parameters", "sec_per_epoch"]].round(4))

##### Plot the best model's confusion matrix

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

activity_names = [
    "Walking",
    "Walking Upstairs",
    "Walking Downstairs",
    "Sitting",
    "Standing",
    "Laying"
]

fig, ax = plt.subplots(figsize=(8, 7))

display_matrix = ConfusionMatrixDisplay(
    confusion_matrix=best_overall["confusion_matrix"],
    display_labels=activity_names
)

display_matrix.plot(
    ax=ax,
    cmap="Blues",
    values_format="d",
    xticks_rotation=35,
    colorbar=False
)

ax.set_title(
    f"Best HAR Model: {best_overall['model']} "
    f"(Seed {best_overall['seed']})"
)
ax.set_xlabel("Predicted Activity")
ax.set_ylabel("Actual Activity")

plt.tight_layout()

os.makedirs("problem1/figures", exist_ok=True)

plt.savefig(
    "problem1/figures/har_best_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

## TASK 4: Adding Problem

In [ ]:
# Prepare datasets for all four sequence lengths.
ADDING_LENGTHS = [50, 100, 200, 400]
ADDING_SEEDS = [42, 123, 2026]

adding_data = {}

for T in ADDING_LENGTHS:

    X_train_add, y_train_add = make_adding(10000, T, seed=0)
    X_val_add, y_val_add = make_adding(1000, T, seed=1)
    X_test_add, y_test_add = make_adding(1000, T, seed=2)

    # Verify dataset shapes
    assert X_train_add.shape == (10000, T, 2)
    assert X_val_add.shape == (1000, T, 2)
    assert X_test_add.shape == (1000, T, 2)

    # Verify exactly two markers per sequence
    assert np.all(X_train_add[:, :, 1].sum(axis=1) == 2)
    assert np.all(X_val_add[:, :, 1].sum(axis=1) == 2)
    assert np.all(X_test_add[:, :, 1].sum(axis=1) == 2)

    # Verify the constant-prediction baseline
    baseline_mse = np.mean((y_test_add - 1.0) ** 2)

    adding_data[T] = {
        "X_train": X_train_add,
        "y_train": y_train_add,
        "X_val": X_val_add,
        "y_val": y_val_add,
        "X_test": X_test_add,
        "y_test": y_test_add
    }

    print(f"\nSequence length T = {T}")
    print("Training:", X_train_add.shape)
    print("Validation:", X_val_add.shape)
    print("Testing:", X_test_add.shape)
    print(f"Constant prediction baseline MSE: {baseline_mse:.5f}")

print("\nAll four adding-problem datasets prepared successfully!")

#### Build the regression models

In [ ]:

# Problem 1 - Task 4: Adding Problem Models

import torch
import torch.nn as nn
import numpy as np

ADDING_HIDDEN_SIZE = 64
ADDING_INPUT_SIZE = 2
ADDING_OUTPUT_SIZE = 1

class AddingProblemModel(nn.Module):
  def __init__(self, model_type, input_size=2, hidden_size=64, output_size=1):
      super().__init__()
      self.model_type = model_type.upper()

      if self.model_type == "RNN":
          self.recurrent = nn.RNN(
              input_size,
              hidden_size,
              batch_first=True,
              nonlinearity="tanh"
          )

      elif self.model_type == "LSTM":
          self.recurrent = nn.LSTM(
              input_size,
              hidden_size,
              batch_first=True
          )

      elif self.model_type == "GRU":
          self.recurrent = nn.GRU(
              input_size,
              hidden_size,
              batch_first=True
          )

      else:
          raise ValueError("Unsupported model type")

      self.regressor = nn.Linear(hidden_size, output_size)

  def forward(self, x):

      if self.model_type == "LSTM":
          _, (hidden, cell) = self.recurrent(x)
      else:
          _, hidden = self.recurrent(x)

      # Last layer's final hidden state
      final_hidden = hidden[-1]

      # Return one continuous prediction per sequence
      prediction = self.regressor(final_hidden)

      return prediction.squeeze(-1)


# Verify all three models
for model_type in ["RNN", "LSTM", "GRU"]:
  model = AddingProblemModel(model_type)

  sample_input = torch.randn(128, 50, 2)
  prediction = model(sample_input)

  parameter_count = sum(
      p.numel() for p in model.parameters() if p.requires_grad
  )

  print(f"\n{model_type}")
  print("Output shape:", prediction.shape)
  print("Trainable parameters:", parameter_count)

  assert prediction.shape == (128,)

print("\nAll three regression model checks passed!")

##### Training Configuration

In [ ]:
ADDING_BATCH_SIZE = 128
ADDING_EPOCHS = 20
ADDING_LEARNING_RATE = 1e-3

adding_device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Training device:", adding_device)

#### Training and Evaluation

In [ ]:
# from torch.utils.data import TensorDataset, DataLoader
def set_adding_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)

  if torch.cuda.is_available():
      torch.cuda.manual_seed_all(seed)


def create_adding_loaders(data, seed, batch_size=128):
  X_train = torch.tensor(data["X_train"], dtype=torch.float32)
  y_train = torch.tensor(data["y_train"], dtype=torch.float32)

  X_val = torch.tensor(data["X_val"], dtype=torch.float32)
  y_val = torch.tensor(data["y_val"], dtype=torch.float32)

  X_test = torch.tensor(data["X_test"], dtype=torch.float32)
  y_test = torch.tensor(data["y_test"], dtype=torch.float32)

  generator = torch.Generator()
  generator.manual_seed(seed)

  train_loader = DataLoader(
      TensorDataset(X_train, y_train),
      batch_size=batch_size,
      shuffle=True,
      generator=generator
  )

  val_loader = DataLoader(
      TensorDataset(X_val, y_val),
      batch_size=batch_size,
      shuffle=False
  )

  test_loader = DataLoader(
      TensorDataset(X_test, y_test),
      batch_size=batch_size,
      shuffle=False
  )

  return train_loader, val_loader, test_loader


def evaluate_adding_model(model, loader, device):
  model.eval()
  total_squared_error = 0.0
  total_samples = 0

  with torch.no_grad():
    for X_batch, y_batch in loader:
      X_batch = X_batch.to(device)
      y_batch = y_batch.to(device)

      predictions = model(X_batch)
      squared_error = (predictions - y_batch) ** 2

      total_squared_error += squared_error.sum().item()
      total_samples += y_batch.size(0)

  return total_squared_error / total_samples


def train_adding_model(model, train_loader, val_loader, epochs=20, learning_rate=1e-3, device=adding_device):
    model = model.to(device)

    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    history = {"train_mse": [], "val_mse": []}

    best_val_mse = float("inf")
    best_state = None
    epoch_times = []

    for epoch in range(epochs):
        model.train()

        if device.type == "cuda":
            torch.cuda.synchronize()

        epoch_start = time.perf_counter()

        for X_batch, y_batch in train_loader:
          X_batch = X_batch.to(device)
          y_batch = y_batch.to(device)

          optimizer.zero_grad()

          predictions = model(X_batch)
          loss = criterion(predictions, y_batch)

          loss.backward()
          optimizer.step()

        if device.type == "cuda":
          torch.cuda.synchronize()

        epoch_times.append(time.perf_counter() - epoch_start)

        train_mse = evaluate_adding_model(
            model, train_loader, device
        )

        val_mse = evaluate_adding_model(
            model, val_loader, device
        )

        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)

        if val_mse < best_val_mse:
          best_val_mse = val_mse
          best_state = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_state)

    return model, history, best_val_mse, np.mean(epoch_times)

##### Smoke testing: one RNN experiment

In [ ]:
set_adding_seed(42)

smoke_train, smoke_val, smoke_test = create_adding_loaders(adding_data[50], seed=42, batch_size=ADDING_BATCH_SIZE)

smoke_model = AddingProblemModel("RNN")

smoke_model, smoke_history, smoke_best_val, smoke_epoch_time = (
    train_adding_model(smoke_model, smoke_train, smoke_val, epochs=2, learning_rate=ADDING_LEARNING_RATE, device=adding_device)
)

smoke_test_mse = evaluate_adding_model(
    smoke_model,
    smoke_test,
    adding_device
)

print("Smoke test completed!")
print("Training MSE:", smoke_history["train_mse"])
print("Validation MSE:", smoke_history["val_mse"])
print("Best validation MSE:", smoke_best_val)
print("Test MSE:", smoke_test_mse)
print("Average epoch time:", smoke_epoch_time)

##### Running all 36 experiements (3 Models, 4 Sequence lengths and 3 seeds)

In [ ]:
ADDING_MODELS = ["RNN", "LSTM", "GRU"]

adding_results = []
adding_histories = {}

total_runs = (
    len(ADDING_MODELS)
    * len(ADDING_LENGTHS)
    * len(ADDING_SEEDS)
)

run_number = 0
for T in ADDING_LENGTHS:

  print(f"\n{'='*55}")
  print(f"Sequence length T = {T}")
  print(f"{'='*55}")

  data = adding_data[T]

  for model_type in ADDING_MODELS:

      for seed in ADDING_SEEDS:

          run_number += 1

          print(
              f"\nRun {run_number}/{total_runs}: "
              f"{model_type}, T={T}, seed={seed}"
          )

          set_adding_seed(seed)

          train_loader, val_loader, test_loader = (
              create_adding_loaders(
                  data,
                  seed=seed,
                  batch_size=ADDING_BATCH_SIZE
              )
          )

          model = AddingProblemModel(
              model_type,
              hidden_size=ADDING_HIDDEN_SIZE
          )

          params = sum(
              p.numel()
              for p in model.parameters()
              if p.requires_grad
          )

          trained_model, history, best_val_mse, sec_per_epoch = (
              train_adding_model(
                  model,
                  train_loader,
                  val_loader,
                  epochs=ADDING_EPOCHS,
                  learning_rate=ADDING_LEARNING_RATE,
                  device=adding_device
              )
          )

          test_mse = evaluate_adding_model(
              trained_model,
              test_loader,
              adding_device
          )

          adding_histories[(T, model_type, seed)] = history

          result = {
              "model": model_type,
              "seq_len": T,
              "seed": seed,
              "test_mse": test_mse,
              "best_val_mse": best_val_mse,
              "params": params,
              "sec_per_epoch": sec_per_epoch,
              "solved": test_mse < 0.05
          }

          adding_results.append(result)

          print(
              f"Test MSE: {test_mse:.6f} | "
              f"Best validation MSE: {best_val_mse:.6f} | "
              f"Time/epoch: {sec_per_epoch:.3f}s | "
              f"Solved: {test_mse < 0.05}"
          )

print("\nAll experiments completed.")
print("Total completed runs:", len(adding_results))

#### Calculate the mean and sample standard deviation

In [ ]:
adding_df = pd.DataFrame(adding_results)
adding_summary = (
    adding_df
    .groupby(["model", "seq_len"])
    .agg(
        test_mse_mean=("test_mse", "mean"),
        test_mse_std=("test_mse", "std"),
        solved_runs=("solved", "sum"),
        avg_sec_per_epoch=("sec_per_epoch", "mean")
    )
    .reset_index()
)

adding_summary["test_mse_mean_std"] = adding_summary.apply(
    lambda row: (
        f'{row["test_mse_mean"]:.5f} '
        f'± {row["test_mse_std"]:.5f}'
    ),
    axis=1
)

print("ADDING PROBLEM: TEST MSE SUMMARY")
display(
    adding_summary[
        ["model", "seq_len", "test_mse_mean_std",
         "solved_runs", "avg_sec_per_epoch"]
    ].sort_values(["seq_len", "model"])
)

print("\nSOLVED RUNS (test MSE < 0.05)")
solved_table = adding_summary.pivot(
    index="model",
    columns="seq_len",
    values="solved_runs"
)

display(solved_table)

##### Plot T=200 learning curves

In [ ]:
T = 200
epochs = np.arange(1, ADDING_EPOCHS + 1)

plt.figure(figsize=(11, 6))

for model_type in ADDING_MODELS:

    train_curves = np.array([
        adding_histories[(T, model_type, seed)]["train_mse"]
        for seed in ADDING_SEEDS
    ])

    val_curves = np.array([
        adding_histories[(T, model_type, seed)]["val_mse"]
        for seed in ADDING_SEEDS
    ])

    train_mean = train_curves.mean(axis=0)
    train_std = train_curves.std(axis=0)

    val_mean = val_curves.mean(axis=0)
    val_std = val_curves.std(axis=0)

    plt.plot(
        epochs, train_mean,
        label=f"{model_type} - Train",
        linewidth=2
    )

    plt.fill_between(
        epochs,
        train_mean - train_std,
        train_mean + train_std,
        alpha=0.12
    )

    plt.plot(
        epochs, val_mean,
        linestyle="--",
        label=f"{model_type} - Validation",
        linewidth=2
    )

    plt.fill_between(
        epochs,
        val_mean - val_std,
        val_mean + val_std,
        alpha=0.12
    )

baseline_T200 = np.mean(
    (1.0 - adding_data[200]["y_test"]) ** 2
)

plt.axhline(
    baseline_T200,
    linestyle=":",
    linewidth=2,
    label=f"Constant baseline ({baseline_T200:.4f})"
)

plt.xlabel("Epoch")
plt.ylabel("Mean Squared Error")
plt.title("Adding Problem Learning Curves (T=200)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()

os.makedirs("problem1/figures", exist_ok=True)

figure_path = "problem1/figures/adding_learning_curves_T200.png"
plt.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print("Figure saved to:", figure_path)
print("T=200 test baseline MSE:", baseline_T200)

In [ ]:
results_path = "results.csv"

required_columns = [
    "problem",
    "model",
    "seed",
    "seq_len",
    "test_acc",
    "macro_f1",
    "test_mse",
    "params",
    "sec_per_epoch",
    "val_loss_nats",
    "bits_per_char",
    "notes"
]

adding_csv_rows = []

for result in adding_results:
    adding_csv_rows.append({
        "problem": "P1",
        "model": f"Adding-{result['model']}",
        "seed": result["seed"],
        "seq_len": result["seq_len"],
        "test_acc": np.nan,
        "macro_f1": np.nan,
        "test_mse": result["test_mse"],
        "params": result["params"],
        "sec_per_epoch": result["sec_per_epoch"],
        "val_loss_nats": np.nan,
        "bits_per_char": np.nan,
        "notes": (
            f"Adding problem; best validation MSE="
            f"{result['best_val_mse']:.6f}; "
            f"solved={result['solved']}"
        )
    })

new_rows = pd.DataFrame(adding_csv_rows, columns=required_columns)

if os.path.exists(results_path):
    existing_results = pd.read_csv(results_path)

    # Retain all existing columns and ensure the required schema exists.
    for column in required_columns:
        if column not in existing_results.columns:
            existing_results[column] = np.nan

    combined_results = pd.concat(
        [existing_results, new_rows],
        ignore_index=True
    )
else:
    combined_results = new_rows.copy()

# Remove previous copies of these exact adding-problem experiment keys.
adding_keys = set(
    zip(
        new_rows["problem"],
        new_rows["model"],
        new_rows["seed"],
        new_rows["seq_len"]
    )
)

existing_keys = list(zip(
    combined_results["problem"],
    combined_results["model"],
    combined_results["seed"],
    combined_results["seq_len"]
))

is_repeated_adding_run = [
    key in adding_keys
    for key in existing_keys
]

# Keep the latest row for each adding-problem key.
combined_results = combined_results.loc[
    ~pd.Series(is_repeated_adding_run, index=combined_results.index)
    | ~combined_results["model"].astype(str).str.startswith("Adding-")
]

combined_results = pd.concat(
    [combined_results, new_rows],
    ignore_index=True
)

combined_results.to_csv(results_path, index=False)

print("Saved results to:", results_path)
print("Adding-problem rows saved:", len(new_rows))
print("Total rows in results.csv:", len(combined_results))

display(
    combined_results[
        combined_results["model"].astype(str).str.startswith("Adding-")
    ].head()
)